# Viral Spread

This notebook builds a susceptible-infected-recovered (SIR) model as an AutoReduce `System` and simulates the full ODE model. The example is intentionally lightweight so it can be used as a documentation smoke test.


## SIR Model

The model is

$$
\begin{aligned}
\dot{S} &= -\beta\frac{SI}{N}, \\
\dot{I} &= \beta\frac{SI}{N} - \gamma I, \\
\dot{R} &= \gamma I.
\end{aligned}
$$

The total population $N = S + I + R$ is conserved by the dynamics.


In [ ]:
import numpy as np
from sympy import Symbol

from autoreduce.system.system import System
from autoreduce.utils.reduction import get_ODE

S = Symbol("S")
I = Symbol("I")
R = Symbol("R")
beta = Symbol("beta")
gamma = Symbol("gamma")
N = Symbol("N")

x = [S, I, R]
f = [
    -beta * S * I / N,
    beta * S * I / N - gamma * I,
    gamma * I,
]

population = 1000.0
system = System(
    x,
    f,
    params=[beta, gamma, N],
    params_values=[0.3, 0.08, population],
    x_init=[990.0, 10.0, 0.0],
    C=np.eye(3),
)


## Simulate The Full Model

The ODE solver returns one row per time point and one column per state.


In [ ]:
timepoints = np.linspace(0.0, 160.0, 41)
ode = get_ODE(system, timepoints)
solution = ode.solve_system()

solution.shape


## Check Population Conservation

The total population should remain constant up to numerical integration error.


In [ ]:
total_population = solution.sum(axis=1)
max_conservation_error = np.max(np.abs(total_population - population))

final_state = dict(zip([str(state) for state in system.x], solution[-1]))
max_conservation_error, final_state
